In [1]:
FRAMEWORK = 'pyspark'

# Proyecto Big Data - PySpark

## 0. Instalación, entorno y acceso a los datos

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q pyspark pyarrow pandas gcsfs
print('Entorno:', 'Google Colab' if IN_COLAB else 'local')

Entorno: local


In [3]:
from pathlib import Path

# El dataset se lee directamente desde el bucket de Google Cloud Storage,
# que es el data lake del proyecto. De esta forma el procesamiento consume
# los datos del bucket y no una copia local.
BUCKET = 'gs://bank-segmentation-bigdata-data'
DATA_PATH = f'{BUCKET}/raw/bank_transactions.csv'

if IN_COLAB:
    # Autenticación necesaria para que Colab pueda leer del bucket.
    from google.colab import auth
    auth.authenticate_user()

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / FRAMEWORK
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset: {DATA_PATH}')
print(f'Resultados: {OUTPUT_DIR}')

Dataset: gs://bank-segmentation-bigdata-data/raw/bank_transactions.csv
Resultados: /home/neo/nb2/entrega_modin_pyspark/outputs/pyspark


## 1. SparkSession

In [4]:
import math
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)

spark = SparkSession.builder.master('local[*]').appName(
    'BankCustomerSegmentationColab'
).config('spark.sql.shuffle.partitions', '8').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)
print('Master:', spark.sparkContext.master)
print('Paralelismo:', spark.sparkContext.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/23 02:30:46 INFO SparkEnv: Registering MapOutputTracker
26/09/23 02:30:47 INFO SparkEnv: Registering BlockManagerMaster
26/09/23 02:30:47 INFO SparkEnv: Registering BlockManagerMasterHeartbeat


26/09/23 02:30:47 INFO SparkEnv: Registering OutputCommitCoordinator


Spark: 3.5.3
Master: local[*]
Paralelismo: 4


## 2. Lectura con esquema explícito

In [5]:
AMOUNT = 'TransactionAmount (INR)'
BALANCE = 'CustAccountBalance'
schema = StructType([
    StructField('TransactionID', StringType(), False),
    StructField('CustomerID', StringType(), True),
    StructField('CustomerDOB', StringType(), True),
    StructField('CustGender', StringType(), True),
    StructField('CustLocation', StringType(), True),
    StructField(BALANCE, DoubleType(), True),
    StructField('TransactionDate', StringType(), True),
    StructField('TransactionTime', LongType(), True),
    StructField(AMOUNT, DoubleType(), True),
])
raw = spark.read.option('header', True).option(
    'nullValue', 'nan'
).schema(schema).csv(str(DATA_PATH))
print('Filas:', raw.count())
raw.printSchema()
raw.show(5, truncate=False)

Filas: 1048567
root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerDOB: string (nullable = true)
 |-- CustGender: string (nullable = true)
 |-- CustLocation: string (nullable = true)
 |-- CustAccountBalance: double (nullable = true)
 |-- TransactionDate: string (nullable = true)
 |-- TransactionTime: long (nullable = true)
 |-- TransactionAmount (INR): double (nullable = true)



+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+
|TransactionID|CustomerID|CustomerDOB|CustGender|CustLocation|CustAccountBalance|TransactionDate|TransactionTime|TransactionAmount (INR)|
+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+
|T1           |C5841053  |10/1/94    |F         |JAMSHEDPUR  |17819.05          |2/8/16         |143207         |25.0                   |
|T2           |C2142763  |4/4/57     |M         |JHAJJAR     |2270.69           |2/8/16         |141858         |27999.0                |
|T3           |C4417068  |26/11/96   |F         |MUMBAI      |17874.44          |2/8/16         |142712         |459.0                  |
|T4           |C5342380  |14/9/73    |F         |MUMBAI      |866503.21         |2/8/16         |142714         |2060.0                 |
|T5           |C9031234  |24/3/88 

## 3. Preparación común

In [6]:
transaction_date = F.to_date(F.col('TransactionDate'), 'd/M/yy')
dob_day = F.expr("try_cast(element_at(split(CustomerDOB, '/'), 1) as int)")
dob_month = F.expr("try_cast(element_at(split(CustomerDOB, '/'), 2) as int)")
dob_year_raw = F.expr("try_cast(element_at(split(CustomerDOB, '/'), 3) as int)")
birth_year = F.when(
    dob_year_raw < 100,
    1900 + dob_year_raw + F.when(
        dob_year_raw <= (F.year(transaction_date) % 100), 100
    ).otherwise(0),
).otherwise(dob_year_raw)
birthday_not_reached = F.when(
    (F.month(transaction_date) < dob_month)
    | ((F.month(transaction_date) == dob_month)
       & (F.dayofmonth(transaction_date) < dob_day)), 1
).otherwise(0)
age_raw = F.year(transaction_date) - birth_year - birthday_not_reached
time_text = F.lpad(F.col('TransactionTime').cast('string'), 6, '0')
hour = F.substring(time_text, 1, 2).cast('int')

base = raw.withColumn('TransactionDateParsed', transaction_date).withColumn(
    'LocationNormalized',
    F.upper(F.regexp_replace(F.col('CustLocation'), r'^[\s\p{Z}]+|[\s\p{Z}]+$', ''))
).withColumn('Age', F.when(age_raw.between(18, 100), age_raw)).withColumn(
    'Hour', hour
).withColumn(
    'TimeBand',
    F.when(hour.between(0, 5), 'Madrugada')
    .when(hour.between(6, 11), 'Manana')
    .when(hour.between(12, 17), 'Tarde')
    .when(hour.between(18, 23), 'Noche')
    .otherwise('Invalida')
).withColumn(
    'AgeRange',
    F.when(F.col('Age').between(18, 25), '18-25')
    .when(F.col('Age').between(26, 35), '26-35')
    .when(F.col('Age').between(36, 45), '36-45')
    .when(F.col('Age').between(46, 60), '46-60')
    .when(F.col('Age').between(61, 100), '61-100')
    .otherwise('Desconocido')
).cache()
base.count()
clean = base.dropDuplicates(['TransactionID']).dropDuplicates([
    'CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT
]).cache()
print('Filas preparadas:', clean.count())

26/09/23 02:31:04 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


26/09/23 02:31:31 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


26/09/23 02:31:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Filas preparadas: 1048567


## Pregunta 1. Diagnóstico de calidad de datos

In [7]:
%%time
row_count = raw.count()
null_counts = raw.agg(*[
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw.columns
]).first().asDict()
q01 = pd.DataFrame({
    'column': raw.columns,
    'null_count': [null_counts[c] for c in raw.columns],
})
q01['null_percent'] = (q01['null_count'] * 100 / row_count).round(4)
q01.to_csv(OUTPUT_DIR / 'q01_calidad_datos.csv', index=False)
display(q01)

,column,null_count,null_percent
0,TransactionID,0,0.0000
1,CustomerID,0,0.0000
2,CustomerDOB,3397,0.3240
3,CustGender,1100,0.1049
4,CustLocation,151,0.0144
5,CustAccountBalance,2369,0.2259
6,TransactionDate,0,0.0000
7,TransactionTime,0,0.0000
8,TransactionAmount (INR),0,0.0000


CPU times: user 70.5 ms, sys: 9.48 ms, total: 80 ms
Wall time: 4.5 s


## Pregunta 2. Detección y tratamiento de duplicados

In [8]:
%%time
def duplicate_excess(columns):
    return int(raw.groupBy(*columns).count().filter(F.col('count') > 1).agg(
        F.coalesce(F.sum(F.col('count') - 1), F.lit(0)).alias('duplicates')
    ).first()['duplicates'])

q02 = pd.DataFrame({
    'criterion': [
        'TransactionID',
        'CustomerID+TransactionDate+Amount',
        'CustomerID+TransactionDate+Time+Amount',
    ],
    'duplicate_rows': [
        duplicate_excess(['TransactionID']),
        duplicate_excess(['CustomerID', 'TransactionDate', AMOUNT]),
        duplicate_excess(['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT]),
    ],
    'treatment': [
        'Eliminar repetidos',
        'Conservar y revisar: ocurren a horas distintas',
        'Eliminar repetidos exactos del evento',
    ],
})
q02.to_csv(OUTPUT_DIR / 'q02_duplicados.csv', index=False)
display(q02)

,criterion,duplicate_rows,treatment
0,TransactionID,0,Eliminar repetidos
1,CustomerID+TransactionDate+Amount,31,Conservar y revisar: ocurren a horas distintas
2,CustomerID+TransactionDate+Time+Amount,0,Eliminar repetidos exactos del evento


CPU times: user 60.9 ms, sys: 11.5 ms, total: 72.4 ms
Wall time: 13.8 s


## Pregunta 3. Edad exacta y rangos etarios

In [9]:
%%time
q03 = clean.groupBy('AgeRange').agg(
    F.count('TransactionID').alias('transactions'),
    F.round(F.avg('Age'), 2).alias('mean_age'),
).orderBy('AgeRange')
q03.toPandas().to_csv(OUTPUT_DIR / 'q03_edades.csv', index=False)
q03.show(truncate=False)

+-----------+------------+--------+
|AgeRange   |transactions|mean_age|
+-----------+------------+--------+
|18-25      |295117      |23.08   |
|26-35      |482990      |29.62   |
|36-45      |142585      |39.43   |
|46-60      |50856       |51.03   |
|61-100     |14303       |67.1    |
|Desconocido|62716       |NULL    |
+-----------+------------+--------+

CPU times: user 211 ms, sys: 117 ms, total: 328 ms
Wall time: 3.66 s


## Pregunta 4. Franja horaria y ubicación normalizada

In [10]:
%%time
q04 = clean.groupBy('TimeBand').agg(
    F.count('TransactionID').alias('transactions'),
    F.countDistinct('LocationNormalized').alias('unique_locations'),
).orderBy(F.desc('transactions'), F.asc('TimeBand'))
q04.toPandas().to_csv(OUTPUT_DIR / 'q04_franja_ubicacion.csv', index=False)
q04.show(truncate=False)

+---------+------------+----------------+
|TimeBand |transactions|unique_locations|
+---------+------------+----------------+
|Noche    |436179      |7246            |
|Tarde    |390884      |7146            |
|Manana   |174473      |4889            |
|Madrugada|47031       |2208            |
+---------+------------+----------------+

CPU times: user 18.2 ms, sys: 9.6 ms, total: 27.8 ms
Wall time: 4.25 s


## Pregunta 5. Outliers por percentiles 1 y 99

In [11]:
%%time
balance_p01, balance_p99 = clean.approxQuantile(BALANCE, [0.01, 0.99], 0.001)
amount_p01, amount_p99 = clean.approxQuantile(AMOUNT, [0.01, 0.99], 0.001)
counts = clean.agg(
    F.sum(F.when((F.col(BALANCE) < balance_p01) | (F.col(BALANCE) > balance_p99), 1).otherwise(0)).alias('balance_outliers'),
    F.sum(F.when((F.col(AMOUNT) < amount_p01) | (F.col(AMOUNT) > amount_p99), 1).otherwise(0)).alias('amount_outliers'),
).first()
q05 = pd.DataFrame({
    'variable': [BALANCE, AMOUNT],
    'p01': [balance_p01, amount_p01],
    'p99': [balance_p99, amount_p99],
    'outlier_rows': [counts['balance_outliers'], counts['amount_outliers']],
})
q05.to_csv(OUTPUT_DIR / 'q05_outliers.csv', index=False)
display(q05)

,variable,p01,p99,outlier_rows
0,CustAccountBalance,3.13,1552324.76,21128
1,TransactionAmount (INR),8.02,19750.00,21913


CPU times: user 58.2 ms, sys: 11.6 ms, total: 69.8 ms
Wall time: 4.8 s


## Pregunta 6. Balance y monto promedio por género y edad

In [12]:
%%time
q06 = clean.filter(F.col('CustGender').isNotNull() & F.col('Age').isNotNull()).groupBy(
    'CustGender', 'AgeRange'
).agg(
    F.count('TransactionID').alias('transactions'),
    F.round(F.avg(BALANCE), 2).alias('avg_balance'),
    F.round(F.avg(AMOUNT), 2).alias('avg_amount'),
).orderBy('CustGender', 'AgeRange')
q06.toPandas().to_csv(OUTPUT_DIR / 'q06_genero_edad.csv', index=False)
q06.show(20, truncate=False)

+----------+--------+------------+-----------+----------+
|CustGender|AgeRange|transactions|avg_balance|avg_amount|
+----------+--------+------------+-----------+----------+
|F         |18-25   |94727       |37858.98   |1007.18   |
|F         |26-35   |125778      |84128.28   |1620.86   |
|F         |36-45   |34248       |200026.15  |2323.98   |
|F         |46-60   |14126       |267580.73  |3171.41   |
|F         |61-100  |4210        |695495.59  |3080.99   |
|M         |18-25   |200390      |33768.21   |803.19    |
|M         |26-35   |357212      |84804.58   |1275.34   |
|M         |36-45   |108337      |190350.65  |2155.85   |
|M         |46-60   |36730       |348567.61  |2973.25   |
|M         |61-100  |9942        |649675.48  |3703.29   |
+----------+--------+------------+-----------+----------+

CPU times: user 23 ms, sys: 2.82 ms, total: 25.8 ms
Wall time: 2.39 s


## Pregunta 7. Top 20 ciudades

In [13]:
%%time
q07 = clean.filter(F.col('LocationNormalized').isNotNull()).groupBy(
    'LocationNormalized'
).agg(
    F.count('TransactionID').alias('transactions'),
    F.round(F.sum(AMOUNT), 2).alias('total_amount'),
).orderBy(F.desc('total_amount'), F.asc('LocationNormalized')).limit(20)
q07.toPandas().to_csv(OUTPUT_DIR / 'q07_top_ciudades.csv', index=False)
q07.show(20, truncate=False)

+------------------+------------+--------------+
|LocationNormalized|transactions|total_amount  |
+------------------+------------+--------------+
|MUMBAI            |103596      |1.7968911682E8|
|NEW DELHI         |84928       |1.6070585289E8|
|BANGALORE         |81555       |1.1842484307E8|
|GURGAON           |73818       |1.1209469443E8|
|DELHI             |71019       |1.0622493975E8|
|KOLKATA           |19974       |6.060031018E7 |
|CHENNAI           |30009       |4.463782143E7 |
|NOIDA             |32784       |4.446343311E7 |
|PUNE              |25851       |3.959034875E7 |
|HYDERABAD         |23049       |3.617739443E7 |
|THANE             |21505       |2.715810063E7 |
|GHAZIABAD         |15834       |2.609292893E7 |
|AHMEDABAD         |12264       |2.099287504E7 |
|NAVI MUMBAI       |13080       |2.094954956E7 |
|FARIDABAD         |11318       |1.498607502E7 |
|CHANDIGARH        |9526        |1.491892457E7 |
|JAIPUR            |9921        |1.406685856E7 |
|LUCKNOW           |

## Pregunta 8. Cliente con mayor gasto por ciudad

In [14]:
%%time
spending = clean.filter(
    F.col('LocationNormalized').isNotNull() & F.col('CustomerID').isNotNull()
).groupBy('LocationNormalized', 'CustomerID').agg(
    F.sum(AMOUNT).alias('total_spent'),
    F.count('TransactionID').alias('transactions'),
)
window = Window.partitionBy('LocationNormalized').orderBy(
    F.desc('total_spent'), F.asc('CustomerID')
)
q08 = spending.withColumn('city_rank', F.row_number().over(window)).filter(
    F.col('city_rank') == 1
).withColumn('total_spent', F.round(F.col('total_spent'), 2)).select(
    'LocationNormalized', 'CustomerID', 'total_spent', 'transactions', 'city_rank'
).orderBy(F.desc('total_spent'), F.asc('LocationNormalized'))
q08.toPandas().to_csv(OUTPUT_DIR / 'q08_top_cliente_ciudad.csv', index=False)
print('Ciudades:', q08.count())
q08.show(20, truncate=False)

Ciudades: 9353


+------------------+----------+-----------+------------+---------+
|LocationNormalized|CustomerID|total_spent|transactions|city_rank|
+------------------+----------+-----------+------------+---------+
|GURGAON           |C7319271  |1560034.99 |1           |1        |
|PUNE              |C6677159  |1380002.88 |1           |1        |
|NEW DELHI         |C4141768  |991132.22  |1           |1        |
|MUMBAI            |C8217728  |724122.0   |1           |1        |
|KOLKATA           |C1830891  |720001.16  |1           |1        |
|NOIDA             |C6549785  |600008.32  |1           |1        |
|DELHI             |C4328064  |569500.27  |1           |1        |
|PALAKKARAI TRICHY |C5833636  |557000.73  |1           |1        |
|LUDHIANA          |C8755262  |514320.0   |1           |1        |
|BANGALORE         |C5720892  |500000.0   |1           |1        |
|GATE NO 4 MUMBAI  |C7367184  |455122.0   |1           |1        |
|JALANDHAR         |C6836340  |452400.0   |1           |1     

## Pregunta 9. Serie temporal diaria

In [15]:
%%time
q09 = clean.filter(F.col('TransactionDateParsed').isNotNull()).groupBy(
    'TransactionDateParsed'
).agg(
    F.count('TransactionID').alias('transactions'),
    F.round(F.sum(AMOUNT), 2).alias('total_amount'),
).orderBy('TransactionDateParsed')
q09.toPandas().to_csv(OUTPUT_DIR / 'q09_serie_diaria.csv', index=False)
q09.show(60, truncate=False)

+---------------------+------------+-------------+
|TransactionDateParsed|transactions|total_amount |
+---------------------+------------+-------------+
|2016-08-01           |20438       |2.980181634E7|
|2016-08-02           |20948       |3.046750329E7|
|2016-08-03           |20615       |3.114948367E7|
|2016-08-04           |20682       |3.572271864E7|
|2016-08-05           |21112       |3.483393312E7|
|2016-08-06           |26585       |4.752722782E7|
|2016-08-07           |27261       |4.572777263E7|
|2016-08-08           |21042       |3.012943364E7|
|2016-08-09           |21823       |3.347957048E7|
|2016-08-10           |21649       |3.201630851E7|
|2016-08-11           |21833       |3.186131788E7|
|2016-08-12           |22438       |3.469399433E7|
|2016-08-13           |26921       |4.464547234E7|
|2016-08-14           |25596       |4.57328201E7 |
|2016-08-15           |24171       |4.431309206E7|
|2016-08-16           |20414       |3.247574659E7|
|2016-08-17           |21121   

## Pregunta 10. Ratio gasto/balance y top 1%

In [16]:
%%time
valid = clean.filter(
    F.col(BALANCE).isNotNull() & (F.col(BALANCE) > 0) & F.col(AMOUNT).isNotNull()
).withColumn('spend_balance_ratio', F.col(AMOUNT) / F.col(BALANCE))
n_valid = valid.count()
top_count = math.ceil(n_valid * 0.01)
q10 = valid.orderBy(F.desc('spend_balance_ratio'), F.asc('TransactionID')).limit(
    top_count
).select(
    'TransactionID', 'CustomerID', 'LocationNormalized',
    BALANCE, AMOUNT, 'spend_balance_ratio'
)
q10.toPandas().to_csv(OUTPUT_DIR / 'q10_ratio_top1.csv', index=False)
print('Filas válidas:', n_valid, '| Filas del top 1%:', q10.count())
q10.show(20, truncate=False)

Filas válidas: 1043487 | Filas del top 1%: 10435


+-------------+----------+------------------+------------------+-----------------------+-------------------+
|TransactionID|CustomerID|LocationNormalized|CustAccountBalance|TransactionAmount (INR)|spend_balance_ratio|
+-------------+----------+------------------+------------------+-----------------------+-------------------+
|T742111      |C7323566  |CHANDIGARH        |0.01              |42398.0                |4239800.0          |
|T836117      |C5719489  |KOLAR             |0.01              |25500.0                |2550000.0          |
|T253453      |C3523520  |CHANDIGARH        |0.01              |20000.0                |2000000.0          |
|T652735      |C6038911  |TANK HYDERABAD    |0.01              |17820.0                |1782000.0          |
|T421850      |C2816416  |MUMBAI            |0.01              |15715.0                |1571500.0          |
|T424045      |C7338987  |TANK HYDERABAD    |0.01              |10764.0                |1076400.0          |
|T343016      |C222

In [17]:
spark.stop()